# Limpieza de los datos

**Objetivo:** limpiar el registro académico de estudiantes (una fila por estudiante) para obtener una tabla confiable con la nota final, el resultado del curso y las características de ingreso de cada estudiante.

| | |
|---|---|
| **Entradas** | `Datos/Datos_registro.RData` (objeto `estudiantes_unicos`, 2,720 estudiantes) y `Datos/pensamiento.RData` (objeto `pensamiento_final`, actividades exportadas de Canvas) |
| **Salida** | `Datos/registro_cleaned.csv` (2,325 filas estudiante-intento) |
| **Funciones auxiliares** | [`utils.py`](utils.py): `normalizar_texto`, `determinar_estado_curso`, `determinar_genero`, `aprobado_a_binaria` |
| **Siguiente paso** | [`Limpieza_notas.ipynb`](Limpieza_notas.ipynb) |

### Resumen de decisiones

| # | Problema encontrado | Decisión | Justificación |
|---|---|---|---|
| 1 | `PAA` y `NotaFinal` almacenadas como texto | Convertir a `float64`; `veces_curso` a `Int32`; `Est_ID` a `object` | Son puntajes numéricos; `Est_ID` es un identificador, no una cantidad |
| 2 | 4 valores de PAA fuera del rango válido (200–1600) | Reemplazarlos por la mediana | Son errores de captura; la mediana es robusta a valores extremos |
| 3 | 318 estudiantes (11.7 %) sin ninguna información en el registro | Eliminarlos | Solo 48 tienen sus actividades completas y, aun así, les faltaría PAA, colegio e intento |
| 4 | Estudiante 914 inscrito en hasta 15 secciones en un mismo semestre | Excluirlo | Es un *estudiante de prueba* de Canvas |
| 5 | 112 estudiantes sin nota final (111 retirados, 1 convalidado) | Eliminarlos | No completaron el curso, así que no tienen variable objetivo (~4.6 %) |
| 6 | 5 estudiantes sin `EstadoCurso` | Reconstruirlo con la nota final (≥ 61 → APROBADO) | 61 es la nota mínima de aprobación |
| 7 | Estudiantes sin `Genero` | Imputar con la moda (`Masculino`) | Son muy pocos casos |
| 8 | 395 estudiantes sin PAA (~17 %) | Imputar con la mediana | Eliminarlos implicaría perder demasiada información |
| 9 | Textos con tildes, mayúsculas y espacios inconsistentes | Normalizar con `normalizar_texto` | Evitar que un mismo colegio o título cuente como categorías distintas |
| 10 | `Aprobado` como texto (`Sí`/`No`) | Convertir a binaria (1/0) | Formato requerido por los modelos |
| 11 | Repitentes con una sola fila en el registro | Generar una fila por intento a partir de sus actividades | Cada intento es una observación distinta, con su propio desempeño y resultado |

Primero vamos a empezar limpiando la tabla de estudiantes

In [1]:
# Librerías y funciones auxiliares de limpieza
import pyreadr
import pandas as pd
import math
import seaborn as sns
import re
import unicodedata
from utils import *

In [2]:
# Carga del registro de estudiantes
result = pyreadr.read_r('../../Datos/Datos_registro.RData')
df_students = result['estudiantes_unicos']

In [3]:
# Carga del registro de actividades exportado de Canvas
result = pyreadr.read_r('../../Datos/pensamiento.RData')
df_notas = result['pensamiento_final']

## Análisis inicial

Vamos a ver qué variables están disponibles

In [4]:
# Primeras filas del registro
df_students.head()

,Est_ID,PAA,Colegio,TituloNivelMedio,NotaFinal,Aprobado,Genero,EstadoCurso,veces_curso
0,635.0,1352,INSTITUTO TECNICO DE CAPACITACION Y PRODUCTIVI...,Bachiller En Ciencias Y Letras Y Perito En Ele...,78,Sí,Masculino,APROBADO,1
1,1419.0,1213,CENTRO ESCOLAR KYOOL,Bachiller en Ciencias y Letras,68,Sí,Masculino,APROBADO,1
2,781.0,1413,COLEGIO SUIZO AMERICANO,Bachiller En Ciencias Y Letras Con Orientacion...,86,Sí,Masculino,APROBADO,1
3,977.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,853.0,1414,COLEGIO METROPOLITANO,Bachiller en Ciencias y Letras,69,Sí,Masculino,APROBADO,1


In [5]:
# Dimensiones del registro
df_students.shape

(2720, 9)

In [6]:
# Columnas disponibles
for i in df_students.columns:
    print(i)

Est_ID
PAA
Colegio
TituloNivelMedio
NotaFinal
Aprobado
Genero
EstadoCurso
veces_curso


## Análisis de tipo de dato

In [7]:
# Tipos de dato de cada columna
df_students.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2720 entries, 0 to 2719
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Est_ID            2720 non-null   float64
 1   PAA               1992 non-null   object 
 2   Colegio           2402 non-null   object 
 3   TituloNivelMedio  2402 non-null   object 
 4   NotaFinal         2290 non-null   object 
 5   Aprobado          2402 non-null   object 
 6   Genero            2397 non-null   object 
 7   EstadoCurso       2397 non-null   object 
 8   veces_curso       2402 non-null   object 
dtypes: float64(1), object(8)
memory usage: 191.4+ KB


Algunas columnas no tienen un tipo de dato adecuado. `PAA` y `NotaFinal` deben ser numéricas (`float64`) porque representan puntajes. Además, `veces_curso` es un conteo, así que lo convertimos a entero (`Int32`, que admite valores faltantes), y `Est_ID` lo tratamos como texto (`object`) porque es un identificador: no tiene sentido operar con él aritméticamente.

In [8]:
# PAA y nota final como numéricas
df_students[['PAA', 'NotaFinal']] = df_students[['PAA', 'NotaFinal']].astype("float64")

In [9]:
# Número de intentos como entero que admite valores faltantes
df_students['veces_curso'] = df_students['veces_curso'].astype("Int32")

In [10]:
# El identificador del estudiante se trata como texto
df_students['Est_ID'] = df_students['Est_ID'].astype("object")

In [11]:
# Verificación de los nuevos tipos de dato
df_students.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2720 entries, 0 to 2719
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Est_ID            2720 non-null   object 
 1   PAA               1992 non-null   float64
 2   Colegio           2402 non-null   object 
 3   TituloNivelMedio  2402 non-null   object 
 4   NotaFinal         2290 non-null   float64
 5   Aprobado          2402 non-null   object 
 6   Genero            2397 non-null   object 
 7   EstadoCurso       2397 non-null   object 
 8   veces_curso       2402 non-null   Int32  
dtypes: Int32(1), float64(2), object(6)
memory usage: 183.4+ KB


## Validación de rangos

Ahora vamos a verificar que los valores de cada columna numérica estén en un rango adecuado. 

In [12]:
# Valores mínimos y máximos de las columnas numéricas
df_students[['PAA', 'NotaFinal', 'veces_curso']].agg(['min', 'max'])

,PAA,NotaFinal,veces_curso
min,0.0,0.0,1
max,2700.0,100.0,4


La única columna que presenta anomalías es la del PAA: la nota mínima de dicho examen es 200 puntos y la máxima 1,600, pero hay valores de 0 y de más de 2,000 puntos.

In [13]:
# Registros con PAA fuera del rango válido (200 a 1600)
df_students[(df_students['PAA'] < 200) | (df_students['PAA'] > 1600)]

,Est_ID,PAA,Colegio,TituloNivelMedio,NotaFinal,Aprobado,Genero,EstadoCurso,veces_curso
1682,626.0,2300.0,COLEGIO INGLES AMERICANO EAS DE GUATEMALA,Bachiller en Ciencias y Letras,62.0,Sí,Masculino,APROBADO,1
1742,331.0,0.0,CENTRO DE ESTUDIOS WINBRIDGE,Bachiller en Ciencias y Letras,58.0,No,Femenino,REPROBADO,2
1920,2093.0,2460.0,COLEGIO INTERNACIONAL MONTESSORI,Bachiller Bilingue En Ciencias,81.0,Sí,Masculino,APROBADO,1
2598,2556.0,2700.0,COLEGIO METROPOLITANO,Bachiller en Ciencias y Letras,96.0,Sí,Masculino,APROBADO,1


Son solo 4 registros (0, 2,300, 2,460 y 2,700 puntos), que consideramos errores de captura. **Decisión:** reemplazarlos con la mediana del PAA. Usamos la mediana y no la media porque no se ve afectada por valores extremos. La guardamos en `mediana_paa` para reutilizarla más adelante al imputar los PAA faltantes.

In [14]:
# Mediana del PAA, se usa para corregir e imputar
mediana_paa = df_students['PAA'].median()

In [15]:
# Reemplazo de los PAA fuera de rango por la mediana
df_students.loc[[1682, 1742, 1920, 2598], 'PAA'] = mediana_paa

## Análisis de duplicados

Ahora vamos a analizar si hay registros duplicados en el conjunto de datos. Debe existir solo un registro por estudiante. El ID de estudiante es un identificador único para cada estudiante, por lo tanto el número de id's debe ser igual al número de filas del dataset. 

In [16]:
# Cantidad de identificadores únicos
len(df_students['Est_ID'].unique())

2720

Existen 2720 id's únicos y hay 2720 registros, por lo tanto, no existen duplicados. 

## Eliminación de valores faltantes NA

Nos interesan las siguientes variables:

- Nota
- Si el estudiante fue aprobado o no
- La nota en PAA
- El género de los estudiantes
- El número de intento
- El colegio de donde viene

Todas estas variables nos dan la información más importante para el estudio. 

Ahora vamos a ver si existen NA 

Primero, verificamos la dimensionalidad del dataset, esto será de ayuda para determinar si es conveniente eliminar NA's o no

In [17]:
# Dimensiones antes de tratar valores faltantes
print(f'Columnas: {df_students.shape[1]}, filas {df_students.shape[0]} ')

Columnas: 9, filas 2720 


In [18]:
# Valores faltantes por columna
df_students.isna().sum()

Est_ID                0
PAA                 728
Colegio             318
TituloNivelMedio    318
NotaFinal           430
Aprobado            318
Genero              323
EstadoCurso         323
veces_curso         318
dtype: int64

Hay bastantes NA en nuestras variables de interés, aunque un patrón interesante es que en varias columnas se repite el mismo número de NA (318). Verificaremos si se trata de los mismos registros y luego decidiremos qué hacer con ellos.

In [19]:
# Estudiantes sin ninguna información en el registro
df_suspicious_row = df_students[(df_students[['PAA', 'Colegio', 'TituloNivelMedio', 'NotaFinal', 'Aprobado', 'Genero', 'EstadoCurso', 'veces_curso']].isna()).all(axis=1)].copy()

df_suspicious_row.shape[0]

318

Son las mismas columnas las que poseen NA en casi todos los registros, vamos a verificar si esos estudiantes se encuentran en el conjunto de actividades

In [20]:
# Actividades de los estudiantes sin información en el registro
df_notas = df_notas[df_notas['Est_ID'].isin(df_suspicious_row['Est_ID'])].copy()

In [21]:
# Cantidad de actividades encontradas
df_notas.shape

(3611, 20)

Existen 3611 registros con los IDs de los estudiantes sin información en el dataset de actividades.

In [22]:
# Valores faltantes en esas actividades
df_notas.isna().sum()

Est_ID                  0
Actividad               0
Seccion                 0
Ciclo                   0
zona                    0
Anio                    0
TipoCalificacion        0
PuntosPosibles          0
RevisionPares           0
FechaTodoElDia       1795
FechaVencimiento     1332
TipoEntrega             0
Nota                  102
FechaCalificacion       0
FechaEnvio           1996
TipoEnvio            1996
CantComentarios         0
NotaPtos              102
NotaPorciento         103
perteneceAzona          0
dtype: int64

Además, solamente 103 registros no tienen nota, puede ser que podamos reconstruir la información de un estudiante en base a sus actividades registradas, sin embargo, primero hay que verificar si estos registros corresponden a estudiantes de prueba. En el LMS canvas los estudiantes de prueba son registros en donde el personal docente y evaluador puede realizar pruebas antes de calificar las actividades reales de los estudiantes. Un estudiante de prueba puede identificarse cuando este está en muchas secciones al mismo tiempo en un mismo semestre.  

In [23]:
# Secciones por estudiante y semestre para detectar estudiantes de prueba
df_notas.groupby(['Est_ID', 'Anio', 'Ciclo'])['Seccion'].nunique().sort_values(ascending=False)

Est_ID  Anio  Ciclo
914.0   2021  1        15
        2022  1        13
        2020  1        10
2620.0  2021  1         2
1895.0  2020  1         1
                       ..
918.0   2019  1         1
915.0   2022  1         1
910.0   2021  1         1
909.0   2021  1         1
940.0   2019  1         1
Name: Seccion, Length: 324, dtype: int64

Aquí identificamos a un estudiante de prueba, el 914, debido a que este aparece en muchas secciones del primer ciclo de los años 2021 y 2020, vamos a eliminarlo

In [24]:
# Eliminación del estudiante de prueba de Canvas
df_notas = df_notas[df_notas['Est_ID'] != 914.0].copy()

In [25]:
# Actividades restantes
df_notas.shape

(3170, 20)

Después de eliminar al estudiante de prueba quedan 3,170 registros de actividades.

In [26]:
# Estudiantes sin información en el registro
df_notas['Est_ID'].nunique()

317

Hay al menos 317 estudiantes con su información incompleta. 

In [27]:
# Valores faltantes restantes
df_notas.isna().sum()

Est_ID                  0
Actividad               0
Seccion                 0
Ciclo                   0
zona                    0
Anio                    0
TipoCalificacion        0
PuntosPosibles          0
RevisionPares           0
FechaTodoElDia       1556
FechaVencimiento     1305
TipoEntrega             0
Nota                    1
FechaCalificacion       0
FechaEnvio           1565
TipoEnvio            1565
CantComentarios         0
NotaPtos                1
NotaPorciento           2
perteneceAzona          0
dtype: int64

Además, muchos de ellos si tienen nota en sus respectivas actividades. Ahora verificaremos en que año existen más estos registros. 

In [28]:
# Registros por año
df_notas['Anio'].value_counts()

Anio
2021    1027
2019     770
2022     708
2020     665
Name: count, dtype: int64

In [29]:
# Actividades distintas realizadas por cada estudiante
missing_students = df_notas.groupby(['Est_ID', 'Anio', 'Ciclo', 'Seccion'])['Actividad'].nunique().sort_values(ascending=False)
print(missing_students)

Est_ID  Anio  Ciclo  Seccion
2502.0  2021  1      150        34
777.0   2021  1      150        34
1264.0  2021  1      50         34
24.0    2021  1      10         34
1677.0  2021  1      110        34
                                ..
2485.0  2021  1      30          1
2495.0  2021  1      30          1
2517.0  2021  1      30          1
2470.0  2020  1      20          1
2593.0  2020  1      30          1
Name: Actividad, Length: 322, dtype: int64


Existen varios estudiantes que si poseen sus actividades completas en base a lo visto en la limpieza del conjunto de notas, no podemos reconstruir y obtener la información de los estudiantes que no poseen todas sus actividades, sin embargo, si podemos obtener la nota final de los estudiantes que si tienen sus registros completos, vamos a filtrar para solo aquellos que si tengan sus registros completos. Para esto nos vamos a guiar en el número de actividades realizadas en cada año.  

In [30]:
# Estudiantes con todas las actividades de su año (21 en 2019, 35 en 2020, 34 en 2021 y 25 o 26 en 2022)
df_missing = missing_students.reset_index(name='NumActividades')


condicion = (
    ((df_missing['Anio'] == '2019') & (df_missing['NumActividades'] == 21)) |
    ((df_missing['Anio'] == '2020') & (df_missing['NumActividades'] == 35)) |
    ((df_missing['Anio'] == '2021') & (df_missing['NumActividades'] == 34)) |
    ((df_missing['Anio'] == '2022') & (df_missing['Seccion'] == '120') & (df_missing['NumActividades'] == 25)) |
    ((df_missing['Anio'] == '2022') & (df_missing['Seccion'] != '120') & (df_missing['NumActividades'] == 26))
)

df_completos = df_missing[condicion]

In [31]:
# Cantidad de estudiantes con actividades completas
df_completos.shape

(48, 5)

Solo 48 estudiantes (≈2 % del total) poseen sus notas completas y, aun para ellos, no tenemos su PAA, colegio, título, ni si repitieron el curso. **Decisión:** eliminar los 318 estudiantes sin información en el registro (11.7 % del total), ya que no es posible reconstruir sus características de ingreso.

In [32]:
# Eliminación de estudiantes sin ninguna información en el registro
df_students_clean = df_students.dropna(subset = ['PAA', 'Colegio', 'TituloNivelMedio', 'NotaFinal', 'Aprobado', 'Genero', 'EstadoCurso', 'veces_curso'], how= 'all').copy()

In [33]:
# Valores faltantes después de la eliminación
df_students_clean.isna().sum()

Est_ID                0
PAA                 410
Colegio               0
TituloNivelMedio      0
NotaFinal           112
Aprobado              0
Genero                5
EstadoCurso           5
veces_curso           0
dtype: int64

In [34]:
# Filas restantes
print(f'Quedaron {df_students_clean.shape[0]} filas')

Quedaron 2402 filas


Vamos a ver si las 5 filas en donde no hay un Estado del curso o el número de intento pertencen al conjunto de filas en donde no hay valores para la nota final y el PAA

In [35]:
# Verifica si los faltantes de estado de curso coinciden con los de PAA y nota
df_students_clean[(df_students_clean[['PAA', 'NotaFinal', 'Genero', 'EstadoCurso', 'veces_curso']].isna()).all(axis=1)].shape[0]

0

No se trata de los mismos registros , vamos a evaluar cada columna individualmente. Empezamos con la nota final 

In [36]:
# Estado del curso de los estudiantes sin nota final
df_students_clean[df_students_clean['NotaFinal'].isna()]['EstadoCurso'].value_counts()

EstadoCurso
CURSO RETIRADO    111
CONVALIDADO         1
Name: count, dtype: int64

Los estudiantes con NA en este campo se retiraron o convalidaron el curso, como no hay información suficiente sobre ellos, vamos a eliminarlos, ya que solamente representan un 4.6% de la información total. 

In [37]:
# Eliminación de estudiantes que retiraron o convalidaron el curso
df_students_clean = df_students_clean.drop(
    df_students_clean[df_students_clean['EstadoCurso'].isin(['CURSO RETIRADO', 'CONVALIDADO'])].index
).copy()

Ahora analizamos el estado de curso. 

In [38]:
# Notas finales de los estudiantes sin estado de curso
df_students_clean[df_students_clean['EstadoCurso'].isna()]['NotaFinal'].unique()

array([64., 41., 68., 53., 44.])

In [39]:
# Índices de los estudiantes sin estado de curso
na_estado_row = list(df_students_clean[df_students_clean['EstadoCurso'].isna()].index)

In [40]:
# Estados de curso existentes
df_students_clean['EstadoCurso'].unique()

array(['APROBADO', 'REPROBADO', nan], dtype=object)

In [41]:
# Estado del curso calculado a partir de la nota final
df_students_clean.loc[na_estado_row, 'EstadoCurso'] = df_students_clean.loc[na_estado_row].apply(determinar_estado_curso, axis=1)

In [42]:
# Verificación de los estados de curso
df_students_clean['EstadoCurso'].unique()

array(['APROBADO', 'REPROBADO'], dtype=object)

Para los estudiantes sin género registrado usamos la función `determinar_genero` de `utils.py`, que les asigna `Masculino`, la categoría más frecuente (imputación por moda). Son muy pocos registros, por lo que el efecto sobre el análisis es mínimo.

In [43]:
# Imputación del género faltante
df_students_clean['Genero'] = df_students_clean['Genero'].apply(determinar_genero)

In [44]:
# Valores faltantes restantes
df_students_clean.isna().sum()

Est_ID                0
PAA                 395
Colegio               0
TituloNivelMedio      0
NotaFinal             0
Aprobado              0
Genero                0
EstadoCurso           0
veces_curso           0
dtype: int64

In [45]:
# Categorías de género
df_students_clean['Genero'].unique()

array(['Masculino', 'Femenino'], dtype=object)

Ahora imputamos el PAA faltante con la mediana (`mediana_paa`). Son 395 estudiantes (≈17 % de los registros restantes): eliminarlos supondría perder demasiada información, y la mediana es una estimación robusta del puntaje típico.

In [46]:
# Imputación del PAA faltante con la mediana
df_students_clean['PAA'] = df_students_clean['PAA'].fillna(mediana_paa)

In [47]:
# Verificación de que no quedan valores faltantes
df_students_clean.isna().sum()

Est_ID              0
PAA                 0
Colegio             0
TituloNivelMedio    0
NotaFinal           0
Aprobado            0
Genero              0
EstadoCurso         0
veces_curso         0
dtype: int64

Y ahora, limpiamos las cadenas de texto. Eliminaremos comas, acentos, espacios en blanco innecesarios, etc. Dejaremos todo en mayúscula



In [48]:
# Normalización de textos: mayúsculas, sin tildes ni signos
df_students_clean['Colegio'] = df_students_clean['Colegio'].apply(normalizar_texto)
df_students_clean['TituloNivelMedio'] = df_students_clean['TituloNivelMedio'].apply(normalizar_texto)
df_students_clean['EstadoCurso'] = df_students_clean['EstadoCurso'].apply(normalizar_texto)
df_students_clean['Genero'] = df_students_clean['Genero'].apply(normalizar_texto)

In [49]:
# Resultado de la normalización
df_students_clean.head()

,Est_ID,PAA,Colegio,TituloNivelMedio,NotaFinal,Aprobado,Genero,EstadoCurso,veces_curso
0,635.0,1352.0,INSTITUTO TECNICO DE CAPACITACION Y PRODUCTIVI...,BACHILLER EN CIENCIAS Y LETRAS Y PERITO EN ELE...,78.0,Sí,MASCULINO,APROBADO,1
1,1419.0,1213.0,CENTRO ESCOLAR KYOOL,BACHILLER EN CIENCIAS Y LETRAS,68.0,Sí,MASCULINO,APROBADO,1
2,781.0,1413.0,COLEGIO SUIZO AMERICANO,BACHILLER EN CIENCIAS Y LETRAS CON ORIENTACION...,86.0,Sí,MASCULINO,APROBADO,1
4,853.0,1414.0,COLEGIO METROPOLITANO,BACHILLER EN CIENCIAS Y LETRAS,69.0,Sí,MASCULINO,APROBADO,1
5,160.0,1414.0,CENTRO ESCOLAR ENTREVALLES,BACHILLER EN CIENCIAS Y LETRAS,63.0,Sí,FEMENINO,APROBADO,1


Finalmente, vamos a convertir la variable "Aprobado" en una variable binaria

In [50]:
# Valores de la columna Aprobado
df_students_clean['Aprobado'].unique()

array(['Sí', 'No'], dtype=object)

In [51]:
# Distribución de aprobados
df_students_clean['Aprobado'].value_counts()

Aprobado
Sí    2001
No     287
Name: count, dtype: int64

In [52]:
# Conversión de Aprobado a variable binaria
df_students_clean['Aprobado'] = df_students_clean['Aprobado'].apply(aprobado_a_binaria)

In [53]:
# Resultado de la conversión
df_students_clean.head()

,Est_ID,PAA,Colegio,TituloNivelMedio,NotaFinal,Aprobado,Genero,EstadoCurso,veces_curso
0,635.0,1352.0,INSTITUTO TECNICO DE CAPACITACION Y PRODUCTIVI...,BACHILLER EN CIENCIAS Y LETRAS Y PERITO EN ELE...,78.0,1,MASCULINO,APROBADO,1
1,1419.0,1213.0,CENTRO ESCOLAR KYOOL,BACHILLER EN CIENCIAS Y LETRAS,68.0,1,MASCULINO,APROBADO,1
2,781.0,1413.0,COLEGIO SUIZO AMERICANO,BACHILLER EN CIENCIAS Y LETRAS CON ORIENTACION...,86.0,1,MASCULINO,APROBADO,1
4,853.0,1414.0,COLEGIO METROPOLITANO,BACHILLER EN CIENCIAS Y LETRAS,69.0,1,MASCULINO,APROBADO,1
5,160.0,1414.0,CENTRO ESCOLAR ENTREVALLES,BACHILLER EN CIENCIAS Y LETRAS,63.0,1,FEMENINO,APROBADO,1


In [54]:
# Distribución de la variable binaria
df_students_clean['Aprobado'].value_counts()

Aprobado
1    2001
0     287
Name: count, dtype: int64

Ahora vamos a agregar el año en el que el estudiante llevó el curso y a calcular el número de intento. El registro original tiene una sola fila por estudiante, pero quienes repitieron el curso aparecen en el conjunto de actividades en varios años; por eso nos basamos en las actividades para reconstruir cada intento.

In [55]:
# Recarga del registro completo de actividades
result = pyreadr.read_r('../../Datos/pensamiento.RData')
df_notas = result['pensamiento_final']

In [56]:
# Actividades de los estudiantes que quedaron en el registro limpio
df_notas = df_notas[df_notas['Est_ID'].isin(df_students_clean['Est_ID'].unique())].copy()

In [57]:
# Dimensiones de las actividades
df_notas.shape

(56541, 20)

Eliminamos actividades duplicadas: una misma combinación (estudiante, actividad, año, sección, ciclo) solo debe aparecer una vez.

In [58]:
# Eliminación de actividades duplicadas
df_notas = df_notas.drop_duplicates(subset=['Est_ID', 'Actividad', 'Anio', 'Seccion', 'Ciclo'], keep = 'first')

In [59]:
# Dimensiones sin duplicados
df_notas.shape

(55587, 20)

Separamos a los estudiantes que llevaron el curso en un solo año (`no_repitentes`) de los que aparecen en más de un año (`repitentes`). A los primeros simplemente les asignamos ese año.

In [60]:
# Años en los que aparece cada estudiante
cuenta_estudiantes = df_notas.groupby('Est_ID')['Anio'].unique()

In [61]:
# Separación entre repitentes y estudiantes con un solo año
repitentes = cuenta_estudiantes[cuenta_estudiantes.apply(len) > 1]
no_repitentes = cuenta_estudiantes[cuenta_estudiantes.apply(len) == 1]

In [62]:
# Año del curso de los estudiantes que no repitieron
no_repitentes = no_repitentes.apply(lambda x: x[0])

In [63]:
# Asignación del año en el registro
df_students_clean['Anio'] = df_students_clean['Est_ID'].map(no_repitentes)

In [64]:
# Dimensiones del registro
df_students_clean.shape

(2288, 10)

Para los **repitentes** generamos una fila por intento:

1. La `NotaFinal` de cada intento se reconstruye sumando los puntos (`NotaPtos`) obtenidos en las actividades de ese año, redondeando hacia arriba (`np.ceil`).
2. `veces_curso` se asigna según el orden cronológico de los intentos (1, 2, 3, …).
3. `Aprobado` y `EstadoCurso` se recalculan con la nota mínima de aprobación (61).
4. Las características que no cambian entre intentos (PAA, colegio, título y género) se copian del registro original.
5. Se reemplazan las filas originales de los repitentes por las nuevas.

**Justificación:** cada intento es una observación distinta, con su propio desempeño y resultado; usar una sola fila por estudiante mezclaría ambos intentos.

In [65]:
# Reconstrucción de los intentos de los repitentes
# 1. Nota final de cada intento: suma de los puntos obtenidos en ese año
notas_intento = df_notas.groupby(['Est_ID', 'Anio'])['NotaPtos'].sum().reset_index(name='NotaFinal')

# 2. Quedarnos solo con los intentos de los repitentes
notas_repitentes = notas_intento[notas_intento['Est_ID'].isin(repitentes.index)].copy()

# 3. veces_curso = orden cronológico del intento
notas_repitentes = notas_repitentes.sort_values(['Est_ID', 'Anio'])
notas_repitentes['veces_curso'] = notas_repitentes.groupby('Est_ID').cumcount() + 1

# Redondeo hacia arriba de la nota final
notas_repitentes['NotaFinal'] = np.ceil(notas_repitentes['NotaFinal'])
# 4. Aprobado / EstadoCurso
notas_repitentes['Aprobado'] = (notas_repitentes['NotaFinal'] >= 61).astype(int)
notas_repitentes['EstadoCurso'] = notas_repitentes['Aprobado'].map({1: 'APROBADO', 0: 'REPROBADO'})

# 5. Pegar columnas constantes (PAA, Colegio, TituloNivelMedio, Genero)
notas_repitentes = notas_repitentes.merge(
    df_students_clean[['Est_ID', 'PAA', 'Colegio', 'TituloNivelMedio', 'Genero']].drop_duplicates('Est_ID'),
    on='Est_ID', how='left'
)[df_students_clean.columns]

# 6. Reemplazar en df_registro: quitar filas viejas de repitentes, pegar las nuevas
df_students_clean = pd.concat([
    df_students_clean[~df_students_clean['Est_ID'].isin(repitentes.index)],
    notas_repitentes
], ignore_index=True)

In [66]:
# Dimensiones del registro con una fila por intento
df_students_clean.shape

(2325, 10)

Finalizamos el proceso con éxito. Guardamos los datos limpios en un csv

In [67]:
# Exportación del registro limpio
df_students_clean.to_csv('../../Datos/registro_cleaned.csv', index=False)